# CorpBrain — Notebook 3: Local Agent Pipeline Test

**Runs 100% locally — no API keys, no internet (after install).**

**What this tests:**
1. Rule-based extractor — finds action items from a transcript
2. Keyword classifier — detects if audio is a meeting  
3. Story point estimator — rule table (type × priority)
4. Context agent — derives project key and sprint context
5. Jira payload builder — professional markdown template
6. Full agent chain without any external service

> Use this notebook to validate your local pipeline before training Kaggle models.

## Step 1 — Setup

In [1]:
import sys, re, json, concurrent.futures
from pathlib import Path

# Locate project root (works from project root OR from notebooks/)
PROJECT_ROOT = Path('.').resolve()
if (PROJECT_ROOT / 'cognitive_service').exists():
    ROOT = PROJECT_ROOT
elif (PROJECT_ROOT.parent / 'cognitive_service').exists():
    ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError("Run this notebook from the project root directory")

# Order matters: agents/ must come before agentic_service/
# so 'from context_agent import ...' resolves to agents/context_agent.py
sys.path.insert(0, str(ROOT / 'agentic_service' / 'agents'))
sys.path.insert(0, str(ROOT / 'agentic_service'))
sys.path.insert(0, str(ROOT / 'cognitive_service'))

print(f"Project root: {ROOT}")
print("sys.path configured ✅")
print("  cognitive_service/ → classifier, extractor")
print("  agentic_service/   → config")
print("  agentic_service/agents/ → all agents")


Project root: /home/aabdellmaksoud/VSCODE/Autonomous-Meeting-Intelligence-Platform
sys.path configured ✅
  cognitive_service/ → classifier, extractor
  agentic_service/   → config
  agentic_service/agents/ → all agents


## Step 2 — Test the Keyword Classifier

In [2]:
# Import classifier directly (no service running needed)
from classifier import _keyword_classify

test_cases = [
    ("Sprint 14 planning. Ahmed will fix the login bug by Friday. Sara will deploy to staging. Blocked by missing credentials.", True),
    ("Good morning team. Daily standup: what did you finish yesterday? Any blockers?", True),
    ("Hello, I'm calling about your car warranty. This is an automated message.", False),
    ("Today's lecture covers machine learning fundamentals and neural networks.", False),
    ("Retrospective: what went well? CI pipeline improved. What to fix? Code reviews too slow.", True),
]

print("Keyword Classifier Results:")
print(f"{'Expected':12} | {'Got':12} | {'Conf':6} | Text")
print("-" * 80)
correct = 0
for text, expected_meeting in test_cases:
    r = _keyword_classify(text)
    got_meeting = r['label'] == 'meeting'
    ok = got_meeting == expected_meeting
    if ok: correct += 1
    icon = "✅" if ok else "❌"
    exp_str = "MEETING" if expected_meeting else "NOT-MEET"
    print(f"{exp_str:12} | {r['label']:12} | {r['confidence']:.0%}  | {icon} {text[:50]}...")

print(f"\nAccuracy: {correct}/{len(test_cases)} ({correct/len(test_cases):.0%})")

Keyword Classifier Results:
Expected     | Got          | Conf   | Text
--------------------------------------------------------------------------------
[classifier] Keyword: meeting (meeting=9, not_meeting=0)
MEETING      | meeting      | 92%  | ✅ Sprint 14 planning. Ahmed will fix the login bug b...
[classifier] Keyword: meeting (meeting=5, not_meeting=0)
MEETING      | meeting      | 85%  | ✅ Good morning team. Daily standup: what did you fin...
[classifier] Keyword: not_meeting (meeting=0, not_meeting=2)
NOT-MEET     | not_meeting  | 66%  | ✅ Hello, I'm calling about your car warranty. This i...
[classifier] Keyword: not_meeting (meeting=1, not_meeting=1)
NOT-MEET     | not_meeting  | 60%  | ✅ Today's lecture covers machine learning fundamenta...
[classifier] Keyword: meeting (meeting=4, not_meeting=0)
MEETING      | meeting      | 78%  | ✅ Retrospective: what went well? CI pipeline improve...

Accuracy: 5/5 (100%)


## Step 3 — Test the Rule-Based Extractor

In [3]:
from extractor import _rule_based_extract

transcript = (
    "Good morning everyone. Sprint 14 planning meeting. "
    "Ahmed will complete the React Native setup by Wednesday. "
    "Sara needs to fix the authentication bug — it's blocking all users. "
    "Omar will review all pull requests before end of day. "
    "We decided to deploy to staging on Monday for the client demo. "
    "Main blocker: the CI pipeline is taking 45 minutes per run. "
    "Khaled will investigate and optimize it this week. "
    "The deadline for the payment module is end of sprint."
)

result = _rule_based_extract(transcript)

print("Extractor Output:")
print(f"  meeting_type: {result['meeting_type']}")
print(f"  sprint:       {result['sprint']}")
print(f"  participants: {result['participants']}")
print(f"  action_items: {len(result['action_items'])}")
print()
for i, item in enumerate(result['action_items'], 1):
    print(f"  {i}. [{item['assignee']}] {item['task']}")
print()
print(f"  blockers:     {result['blockers']}")
print(f"  decisions:    {result['decisions']}")

[extractor] Rule-based: 4 action items.
Extractor Output:
  meeting_type: standup
  sprint:       Sprint 14
  participants: ['Omar', 'Sara', 'Khaled', 'Ahmed']
  action_items: 4

  1. [Ahmed] complete the React Native setup by Wednesday
  2. [Omar] review all pull requests before end of day
  3. [Khaled] investigate and optimize it this week
  4. [Sara] fix the authentication bug — it's blocking all users

  blockers:     ['the CI pipeline is taking 45 minutes per run']
  decisions:    ['deploy to staging on Monday for the client demo']


## Step 4 — Test the Story Point Estimator

In [4]:
from estimator import estimate_story_points

tasks = [
    {"task": "Fix critical authentication bug in login flow",  "type": "bug",     "priority": "high"},
    {"task": "Refactor the entire payment module architecture", "type": "feature", "priority": "high"},
    {"task": "Update API documentation",                       "type": "admin",   "priority": "low"},
    {"task": "Add unit tests for dashboard component",         "type": "feature", "priority": "medium"},
    {"task": "Fix typo in welcome email template",             "type": "bug",     "priority": "low"},
    {"task": "Migrate database to new schema",                 "type": "feature", "priority": "high"},
    {"task": "Research GraphQL vs REST for mobile API",        "type": "research","priority": "medium"},
]

print("Story Point Estimates:")
print(f"{'Type':10} | {'Priority':8} | {'Points':6} | Task")
print("-" * 70)
for t in tasks:
    pts = estimate_story_points(t)
    print(f"{t['type']:10} | {t['priority']:8} | {pts:6} | {t['task'][:45]}")

Story Point Estimates:
Type       | Priority | Points | Task
----------------------------------------------------------------------
[estimator] Rule-based: 5 pts (bug/high)
bug        | high     |      5 | Fix critical authentication bug in login flow
[estimator] Rule-based: 13 pts (feature/high)
feature    | high     |     13 | Refactor the entire payment module architectu
[estimator] Rule-based: 1 pts (admin/low)
admin      | low      |      1 | Update API documentation
[estimator] Rule-based: 5 pts (feature/medium)
feature    | medium   |      5 | Add unit tests for dashboard component
[estimator] Rule-based: 1 pts (bug/low)
bug        | low      |      1 | Fix typo in welcome email template
[estimator] Rule-based: 13 pts (feature/high)
feature    | high     |     13 | Migrate database to new schema
[estimator] Rule-based: 5 pts (research/medium)
research   | medium   |      5 | Research GraphQL vs REST for mobile API


## Step 5 — Test the Context Agent

In [5]:
# Test context agent directly (no LangGraph running)
from context_agent import context_agent, AgentState

state: AgentState = {
    "meeting_id":     "test-001",
    "cognitive_data": {
        "project":      "CorpBrain Mobile App",
        "sprint":       "Sprint 14",
        "participants": ["Ahmed", "Sara", "Omar", "Khaled"],
        "decisions":    ["Deploy to staging Monday", "Use React Native"],
        "blockers":     ["CI pipeline too slow", "Missing design specs"],
        "action_items": [],
    },
    "context":        {},
    "tasks":          [],
    "approved_tasks": [],
    "jira_tickets":   [],
    "error":          None,
}

state = context_agent(state)
print("Context Agent Output:")
for k, v in state['context'].items():
    print(f"  {k}: {v}")

[context_agent] project_key=CMA | sprint=Sprint 14 | team=['Ahmed', 'Sara', 'Omar', 'Khaled']
Context Agent Output:
  project_key: CMA
  sprint_name: Sprint 14
  team_members: ['Ahmed', 'Sara', 'Omar', 'Khaled']
  priority_context: Blockers: CI pipeline too slow; Missing design specs
  decisions_this_sprint: ['Deploy to staging Monday', 'Use React Native']
  tech_stack_hints: []
  definition_of_done: not specified


## Step 6 — Full Local Agent Chain (End-to-End)

In [6]:
# ── Full Agent Chain ──────────────────────────────────────────────────────────
# Imports from agentic_service/agents/ (added to sys.path in Cell 1)
from context_agent import context_agent, AgentState
from task_splitter  import task_splitter
from task_agent     import task_agent
from approval       import approval_gate

full_state: AgentState = {
    "meeting_id": "test-001",
    "cognitive_data": {
        "project":      "CorpBrain Mobile App",
        "sprint":       "Sprint 14",
        "participants": ["Ahmed", "Sara", "Omar", "Khaled"],
        "decisions":    ["Deploy to staging on Monday"],
        "blockers":     ["CI pipeline too slow"],
        "action_items": [
            {"task": "Fix CI pipeline performance", "assignee": "Khaled",
             "priority": "high",   "type": "admin",   "context": "45 min runs", "deadline": "this week"},
            {"task": "Complete React Native setup", "assignee": "Ahmed",
             "priority": "high",   "type": "feature", "context": "Sprint 14",   "deadline": "Wednesday"},
            {"task": "Fix authentication bug",      "assignee": "Sara",
             "priority": "high",   "type": "bug",     "context": "Blocking all","deadline": "today"},
            {"task": "Review all open PRs",         "assignee": "Omar",
             "priority": "medium", "type": "admin",   "context": "3 pending",   "deadline": "end of day"},
        ],
        "is_meeting": True,
    },
    "context": {}, "tasks": [], "approved_tasks": [], "jira_tickets": [], "error": None,
}

print("=== FULL AGENT CHAIN ===\n")
print("[1/4] Context Agent...")
full_state = context_agent(full_state)
print(f"  project_key={full_state['context']['project_key']} | sprint={full_state['context']['sprint_name']}\n")

print("[2/4] Task Splitter...")
full_state = task_splitter(full_state)
print(f"  {len(full_state['tasks'])} tasks split\n")

print("[3/4] Task Agent (parallel story points + Jira payloads)...")
full_state = task_agent(full_state)

print("\n[4/4] Approval Gate...")
full_state = approval_gate(full_state)

print("\n=== RESULTS ===")
for t in full_state['tasks']:
    emoji = {"high": "🔴", "medium": "🟡", "low": "🟢"}.get(t['priority'], "⚪")
    print(f"  {emoji} [{t['task_index']}] {t['story_points']}pts | {t['task'][:55]}")
    print(f"       {t['assignee']} | status: {t['status']}")
    print(f"       Jira: {t['jira_payload']['summary'][:60]}")
    print()
print(f"✅ {len(full_state['tasks'])} tasks processed offline")


=== FULL AGENT CHAIN ===

[1/4] Context Agent...
[context_agent] project_key=CMA | sprint=Sprint 14 | team=['Ahmed', 'Sara', 'Omar', 'Khaled']
  project_key=CMA | sprint=Sprint 14

[2/4] Task Splitter...
[task_splitter] Split into 4 tasks
  4 tasks split

[3/4] Task Agent (parallel story points + Jira payloads)...
[task_agent] Task 0: Fix CI pipeline performance...
[estimator] Rule-based: 3 pts (admin/high)
[task_agent] Task 0 done — 3 pts
[task_agent] Task 1: Complete React Native setup...
[estimator] Rule-based: 8 pts (feature/high)
[task_agent] Task 1 done — 8 pts
[task_agent] Task 2: Fix authentication bug...
[estimator] Rule-based: 5 pts (bug/high)
[task_agent] Task 2 done — 5 pts
[task_agent] Task 3: Review all open PRs...
[estimator] Rule-based: 2 pts (admin/medium)
[task_agent] Task 3 done — 2 pts
[task_agent] All 4 tasks processed.

[4/4] Approval Gate...
[approval] 4 tasks waiting for human approval.

=== RESULTS ===
  🔴 [0] 3pts | Fix CI pipeline performance
       Khaled | 